Code using Arabidopsis train and test data (provided along-with in the Github repository)

In [ ]:
# transpose paired test data - Step 1

import pandas as pd

# Load the data
df = pd.read_csv('A1-paired-test.txt', sep='\t')

# Store the original header of the first column
original_header = df.columns[0]

# Set the first column as the index
df = df.set_index(df.columns[0])

# Transpose the DataFrame
transposed_df = df.transpose()

# Reset the index to make the previous index a column
transposed_df = transposed_df.reset_index()

# Rename the 'index' column to the original header
transposed_df = transposed_df.rename(columns={'index': original_header})

# Convert column names to numeric, errors='ignore' preserves original non-numeric
transposed_df.columns = pd.to_numeric(transposed_df.columns, errors='ignore')

# Save the transposed data
transposed_df.to_csv('A1-paired-test_transposed.txt', sep='\t', index=False)

In [ ]:
# transpose paired train data - Step 2

import pandas as pd

# Load the train data
df_train = pd.read_csv('A1-paired-train.txt', sep='\t')

# Store the original header of the first column
original_header_train = df_train.columns[0]

# Set the first column as the index
df_train = df_train.set_index(df_train.columns[0])

# Convert index to string to prevent numeric suffixing during transpose
df_train.index = df_train.index.astype(str)

# Transpose the DataFrame
transposed_df_train = df_train.transpose()

# Reset the index to make the previous index a column
transposed_df_train = transposed_df_train.reset_index()

# Rename the 'index' column to the original header
transposed_df_train = transposed_df_train.rename(columns={'index': original_header_train})

# Save the transposed train data
transposed_df_train.to_csv('A1-paired-train_transposed.txt', sep='\t', index=False)

In [ ]:
# create separate files for each gene in train set - Step 3

import pandas as pd
import os

# Define the input and output directories
input_file = 'A1-paired-train_transposed.txt'
output_dir = 'A1-gene_files'

# Create the output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Read the first line of the input file to get the gene names
with open(input_file, 'r') as f:
    header = f.readline().strip().split('\t')
    gene_names = header[::2]  # Extract gene names (every other element)

# Read the rest of the file into a DataFrame, skipping the header row
df = pd.read_csv(input_file, sep='\t', header=None, skiprows=1)

for j, gene_name in enumerate(gene_names):
    # Calculate column index for 'array' and 'seq' using gene index
    array_col_index = j * 2
    seq_col_index = j * 2 + 1

    output_file = os.path.join(output_dir, f'{gene_name}.txt')

    # Create the data for this gene file
    data = {'train': ['train'] * len(df),
            'gene': [gene_name] * len(df),
            'array': df.iloc[:, array_col_index],
            'seq': df.iloc[:, seq_col_index]}

    # Create a DataFrame from the collected data
    gene_df = pd.DataFrame(data)

    # Save the DataFrame to a new file
    gene_df.to_csv(output_file, sep='\t', index=False, header=False)

In [ ]:
# QC creation of separate gene_files in train set - Step 4

! ls A1-gene_files/* | wc -l

In [ ]:
# create separate files for each gene in test set - Step 5

import pandas as pd
import os

# Define the input and output directories for test files
input_file_test = 'A1-paired-test_transposed.txt'
output_dir_test = 'A1-gene_files_test'

# Create the output directory if it doesn't exist
os.makedirs(output_dir_test, exist_ok=True)

# Read the first line of the input file to get the gene names
with open(input_file_test, 'r') as f:
    header = f.readline().strip().split('\t')
    gene_names = header[::2]  # Extract gene names (every other element)

# Read the rest of the file into a DataFrame, skipping the header row
df_test = pd.read_csv(input_file_test, sep='\t', header=None, skiprows=1)

# Process all genes
for j, gene_name in enumerate(gene_names):
    # Calculate column index for 'array' and 'seq' using gene index
    array_col_index = j * 2
    seq_col_index = j * 2 + 1

    output_file = os.path.join(output_dir_test, f'{gene_name}.txt')

    # Create the data for this gene file, with 'test' in the first column
    data = {'test': ['test'] * len(df_test),  # Change to 'test'
            'gene': [gene_name] * len(df_test),
            'array': df_test.iloc[:, array_col_index],
            'seq': df_test.iloc[:, seq_col_index]}

    # Create a DataFrame from the collected data
    gene_df = pd.DataFrame(data)

    # Save the DataFrame to a new file
    gene_df.to_csv(output_file, sep='\t', index=False, header=False)

In [ ]:
# QC creation of separate gene_files in train set - Step 6

! ls A1-gene_files_test/* | wc -l

In [ ]:
# A1 prompt: combine train and test data for each gene - Step 7

import pandas as pd
import os
import shutil  # For moving files

# Define the input and output directories
gene_files_dir = 'A1-gene_files'
gene_files_test_dir = 'A1-gene_files_test'
output_dir_combined = 'A1-combined-step1'

# Create the output directory if it doesn't exist
os.makedirs(output_dir_combined, exist_ok=True)

# Get the list of gene file names from the 'gene_files' directory
gene_files = [f for f in os.listdir(gene_files_dir) if os.path.isfile(os.path.join(gene_files_dir, f))]

# Iterate through the gene files and concatenate with corresponding test files
for gene_file in gene_files:
    gene_name = os.path.splitext(gene_file)[0]  # Get gene name without extension

    # Construct paths to train and test files
    train_file_path = os.path.join(gene_files_dir, gene_file)
    test_file_path = os.path.join(gene_files_test_dir, gene_file)

    # Construct path to the combined output file
    combined_file_path = os.path.join(output_dir_combined, gene_file)

    # Check if the test file exists
    if os.path.exists(test_file_path):
        # Concatenate train and test files using pandas
        try:
            train_df = pd.read_csv(train_file_path, sep='\t', header=None)
            test_df = pd.read_csv(test_file_path, sep='\t', header=None)
            combined_df = pd.concat([train_df, test_df], ignore_index=True)
            combined_df.to_csv(combined_file_path, sep='\t', index=False, header=False)
        except pd.errors.EmptyDataError:
            print(f"Warning: Skipping empty file: {gene_file}")
    else:
        print(f"{gene_name}")
        # If test file is not found, copy the train file as is (optional)
        # shutil.copy(train_file_path, combined_file_path)

In [ ]:
# remove intermediate folders - Step 8

! rm -rf A1-gene_files A1-gene_files_test

In [ ]:
# number re-formatting - Step 9

import os
import re

# Define the directory containing the files
directory = 'A1-combined-step1'

# Iterate through all files in the directory
for filename in os.listdir(directory):
    if filename.endswith(".txt"):
        filepath = os.path.join(directory, filename)

        # Read the file contents
        with open(filepath, 'r') as f:
            lines = f.readlines()

        # Modify the lines
        modified_lines = []
        for line in lines:
            parts = line.strip().split('\t')  # Split line by tabs
            if len(parts) >= 3:  # Ensure there's a third column
                # Apply the replacement using regex
                parts[2] = re.sub(r'(\d+\.\d+)\.\d+', r'\1', parts[2])
            modified_lines.append('\t'.join(parts) + '\n')  # Join back with tabs

        # Write the modified contents back to the file
        with open(filepath, 'w') as f:
            f.writelines(modified_lines)

print("Reformatting complete.")

In [ ]:
# printing log transformed rmse for predefined test set (for tool comparison): Step 10

import pandas as pd
import numpy as np
import os
import warnings
import sys
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

# Suppress SettingWithCopyWarning
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

# Specify the output file path for average RMSE
output_file_avg = 'A1-results-polynomial-log_transformed_test_set_only.txt' # Updated output filename

def analyze_gene(df, gene_name):

    train_df = df[df['c1'] == 'train'].copy()
    test_df = df[df['c1'] == 'test'].copy()

    # Ensure test set is not empty
    if test_df.empty:
        print(f"Warning: Test set is empty for gene {gene_name}. Skipping analysis.")
        return np.nan, np.nan

    # Apply Log Transformation
    # Convert columns to numeric before applying log transformation, handling errors
    train_df['x'] = pd.to_numeric(train_df['x'], errors='coerce')
    train_df['y'] = pd.to_numeric(train_df['y'], errors='coerce')
    test_df['x'] = pd.to_numeric(test_df['x'], errors='coerce')
    test_df['y'] = pd.to_numeric(test_df['y'], errors='coerce')

    train_df['x_log'] = np.log1p(train_df['x'])
    train_df['y_log'] = np.log1p(train_df['y'])
    test_df['x_log'] = np.log1p(test_df['x'])
    test_df['y_log'] = np.log1p(test_df['y'])


    # Handle infinite values and missing data in training set after transformation
    train_df['y_log'] = train_df['y_log'].replace([np.inf, -np.inf], np.nan)
    train_df['x_log'] = train_df['x_log'].replace([np.inf, -np.inf], np.nan)
    # Drop rows with NaNs from both x_log and y_log for training
    train_df_cleaned = train_df.dropna(subset=['y_log', 'x_log']).copy()


    # Ensure training set is not empty after dropping NaNs
    if train_df_cleaned.empty:
        print(f"Warning: Training set is empty or contains only NaNs for gene {gene_name}. Skipping analysis.")
        return np.nan, np.nan


    # --- Predicting y from x using Polynomial Regression on log-transformed data ---
    try:
        poly = PolynomialFeatures(degree=2)
        X_train_poly = poly.fit_transform(train_df_cleaned[['x_log']])
        y_train_log = train_df_cleaned['y_log']

        # Prepare test data for prediction
        # We need to align test data with the features the model was trained on
        # and handle NaNs in the test set appropriately for prediction.
        # Drop rows from the test set where the predictor variable is NaN (for this specific prediction)
        test_df_y_pred_cleaned = test_df.dropna(subset=['x_log']).copy()

        if test_df_y_pred_cleaned.empty:
            rmse_y = np.nan
        else:
            X_test_poly = poly.transform(test_df_y_pred_cleaned[['x_log']])
            model_y = LinearRegression()
            model_y.fit(X_train_poly, y_train_log) # Train on cleaned train data

            predicted_y_log = model_y.predict(X_test_poly)

            # Calculate RMSE on the log-transformed test data where both actual and predicted values are available
            rmse_y = np.sqrt(np.mean((test_df_y_pred_cleaned['y_log'] - predicted_y_log)**2))

    except Exception as e:
         print(f"Error during y from x prediction for gene {gene_name}: {e}")
         rmse_y = np.nan # Return NaN on error


    # --- Predicting x from y using Polynomial Regression on log-transformed data ---
    try:
        poly_x = PolynomialFeatures(degree=2)
        Y_train_log = train_df_cleaned[['y_log']].copy() # Use cleaned training data
        x_train_log = train_df_cleaned['x_log'].copy() # Use cleaned training data

        # Ensure train data for x prediction is not empty
        if Y_train_log.empty or x_train_log.empty:
             print(f"Warning: Training data for x prediction is empty for gene {gene_name}. Skipping x prediction.")
             return rmse_y, np.nan

        # Prepare test data for prediction
        # Drop rows from the test set where the predictor variable is NaN (for this specific prediction)
        test_df_x_pred_cleaned = test_df.dropna(subset=['y_log']).copy()

        if test_df_x_pred_cleaned.empty:
            rmse_x = np.nan
        else:
            Y_test_poly = poly_x.fit_transform(test_df_x_pred_cleaned[['y_log']]) # Fit and transform on test

            model_x = LinearRegression()
            model_x.fit(Y_train_log, x_train_log) # Train on cleaned training data

            predicted_x_log = model_x.predict(Y_test_poly)

            # Calculate RMSE on the log-transformed test data where both actual and predicted values are available
            rmse_x = np.sqrt(np.mean((test_df_x_pred_cleaned['x_log'] - predicted_x_log)**2))

    except Exception as e:
         print(f"Error during x from y prediction for gene {gene_name}: {e}")
         rmse_x = np.nan # Return NaN on error


    return rmse_y, rmse_x


# Main loop
directory_path = 'A1-combined-step1'
gene_files = [f for f in os.listdir(directory_path) if os.path.isfile(os.path.join(directory_path, f)) and f.endswith('.txt')]

with open(output_file_avg, 'w') as f_avg: # Changed mode to 'w' to overwrite the file
    # Write header to the output file
    f_avg.write("Gene\tTest_Set_RMSE_Y_Prediction\tTest_Set_RMSE_X_Prediction\n") # Updated header

    for gene_file in gene_files:
        print(f"Analyzing gene: {gene_file}")
        sys.stdout.flush()
        file_path = os.path.join(directory_path, gene_file)

        try:
            df = pd.read_csv(file_path, header=None, sep="\t")
            df.columns = ['c1', 'c2', 'x', 'y']
            gene_name = df['c2'].iloc[0]

            # Call the modified analyze_gene function
            rmse_y, rmse_x = analyze_gene(df, gene_name)

            # Print RMSE values
            print(f"  Gene Name: {gene_name}")
            print(f"  Test Set RMSE (y prediction): {rmse_y}")
            print(f"  Test Set RMSE (x prediction): {rmse_x}")
            print("-" * 30)

            # Write results to the output file
            f_avg.write(f"{gene_name}\t{rmse_y}\t{rmse_x}\n")

        except pd.errors.EmptyDataError:
            print(f"Warning: Skipping empty file: {gene_file}")
        except Exception as e:
            print(f"Error processing file {gene_file}: {e}")
        sys.stdout.flush()

print(f"\nTest set RMSE results saved to: {output_file_avg}")
print("Processing complete.")

In [ ]:
# A1 prompt: printing 10-fold cross validation average for log transformed rmse, and respective sample-wise errors: Step 11

import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
import os
import warnings
import sys
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

# Suppress SettingWithCopyWarning
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

# Specify the output file path for average RMSE
output_file_avg = 'A1-results-polynomial-log_transformed.predefined.test_set.txt'

def analyze_gene(df, gene_name):
    kf = KFold(n_splits=2, shuffle=True, random_state=42)
    rmse_y_scores = []
    rmse_x_scores = []

    # Create a list to store fold results DataFrames
    all_fold_results = []

    fold_num = 1
    for train_index, test_index in kf.split(df):

        # train_df = df.iloc[train_index]
        # test_df = df.iloc[test_index]
        train_df = df[df['c1'] == 'train'].copy()
        test_df = df[df['c1'] == 'test'].copy()
        # Apply Log Transformation
        train_df['x'] = np.log1p(train_df['x'])
        train_df['y'] = np.log1p(train_df['y'])
        test_df['x'] = np.log1p(test_df['x'])
        test_df['y'] = np.log1p(test_df['y'])

        # Handle infinite values and missing data
        train_df['y'] = train_df['y'].replace([np.inf, -np.inf], np.nan)
        train_df = train_df.dropna(subset=['y'])

        # Predicting y from x using Polynomial Regression
        poly = PolynomialFeatures(degree=2)
        X_train_poly = poly.fit_transform(train_df[['x']])
        X_test_poly = poly.transform(test_df[['x']])

        model_y = LinearRegression()
        model_y.fit(X_train_poly, train_df['y'])
        test_df['predicted_y'] = model_y.predict(X_test_poly)

        # Predicting x from y using Polynomial Regression
        poly_x = PolynomialFeatures(degree=2)
        Y_train_poly = poly_x.fit_transform(train_df[['y']])
        Y_test_poly = poly_x.transform(test_df[['y']])

        model_x = LinearRegression()
        model_x.fit(Y_train_poly, train_df['x'])
        test_df['predicted_x'] = model_x.predict(Y_test_poly)

        rmse_y = np.sqrt(np.mean((test_df['y'] - test_df['predicted_y'])**2))
        rmse_x = np.sqrt(np.mean((test_df['x'] - test_df['predicted_x'])**2))

        # print(f"Arr Test Errors: {(test_df['predicted_x'] - test_df['x']).tolist()}")
        # print(f"Seq Test Errors: {(test_df['predicted_y'] - test_df['y']).tolist()}")
        # Create a DataFrame for this fold's results
        fold_results = pd.DataFrame({
            'gene_name': [gene_name],
            'fold': [fold_num],
            'rmse_x': [rmse_x],
            'rmse_y': [rmse_y],
            'arr_test_pred': [(test_df['predicted_x']).tolist()],
            'seq_test_pred': [(test_df['predicted_y']).tolist()]
        })

        # Save fold results to a separate file
        output_file_fold = f'A1-results-polynomial-log_transformed-fold{fold_num}.txt'
        fold_results.to_csv(output_file_fold, sep='\t', index=False, header=False, mode='a')

        rmse_y_scores.append(rmse_y)
        rmse_x_scores.append(rmse_x)
        fold_num += 1  # Increment fold counter

    avg_rmse_y = np.mean(rmse_y_scores)
    avg_rmse_x = np.mean(rmse_x_scores)
    return avg_rmse_y, avg_rmse_x


# Main loop
directory_path = '/content/drive/MyDrive/xplat/A1-combined-step1'
gene_files = [f for f in os.listdir(directory_path) if os.path.isfile(os.path.join(directory_path, f)) and f.endswith('.txt')]

# Create output directory for fold-specific results if it doesn't exist
fold_results_dir = '/content/drive/MyDrive/xplat/A1-fold_results'  # Specify full path
os.makedirs(fold_results_dir, exist_ok=True)  # Create directory if it doesn't exist

with open(output_file_avg, 'a') as f_avg:
  for gene_file in gene_files:
    print(f"Analyzing gene: {gene_file}")
    sys.stdout.flush()
    file_path = os.path.join(directory_path, gene_file)
    df = pd.read_csv(file_path, header=None, sep="\t")
    df.columns = ['c1', 'c2', 'x', 'y']
    gene_name = df['c2'].iloc[0]

    # Call analyze_gene with gene_name
    avg_rmse_y, avg_rmse_x = analyze_gene(df, gene_name)

    # Print average RMSE values
    print(f"  Gene Name: {gene_name}")
    print(f"  10-Fold Cross-Validation RMSE (x prediction): {avg_rmse_x}")
    print(f"  10-Fold Cross-Validation RMSE (y prediction): {avg_rmse_y}")
    print("-" * 30)